In [1]:
#first of all will check model availability
import os
import requests

def query(prompt):
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:8b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )
    return res.json()["response"]

print(query("""who are you, kid?"""))

Hey there! I'm Qwen, an AI assistant developed by Alibaba Cloud. I'm here to help with all sorts of tasks, from answering questions to creating content, solving problems, and more. Whether you need help with homework, want to chat about interesting topics, or just need someone to talk to, I'm here for you! What's on your mind? 😊


In [ ]:

from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatOllama(model="qwen3:8b",temperature=0,)
response = llm.invoke("who are you, kid?")
print("Qwen3_8b_local response: ", response.content,"\n")
GROQ_API_KEY = ""
GEMINI_API_KEY = ""

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0,)
response = llm.invoke("who are you, kid?")
print("Groq response:",response.content,"\n")

llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest",api_key=GEMINI_API_KEY,temperature=0,)
response = llm.invoke("who are you, kid?")
print("Gemini response:",response.text)

Qwen3_8b_local response:  Hey there! I'm Qwen, an AI assistant developed by Alibaba Cloud. I'm here to help with all sorts of tasks, from answering questions to creating content, solving problems, and more. Whether you need help with homework, want to chat about interesting topics, or just need someone to talk to, I'm here for you! What's on your mind? 😊 

Groq response: I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI." 

Gemini response: Not a kid, actually—just an AI here to help you out. What's on your mind?


We will make simple __agent__ that takes a job posting or service description as input and performs a three-level classification:

* Job type: project-based work or permanent position
* Job category: one of predefined professions
* Search type: whether the user is looking for a job or looking for a specialist

The result is a structured JSON object for each classification.

Following the LangGraph principles, we build a state graph consisting of four sequential nodes:

Input description  (with tool)
      ↓  
Job type classification node  
      ↓  
Job category classification node   
(with additional query if needed)   
      ↓  
Search type classification node   
      ↓  
JSON output  (with tool)

In [3]:
# and resume with LangGraph
import json
import pandas as pd
from typing import TypedDict
from enum import Enum

from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import ToolNode, tools_condition


In [4]:
# define classification categories
CATEGORIES = [
    "Data Scientist",
    "Data Analyst",
    "Data Engineer",
    "MLOps Engineer",
    "Business Analyst",
    "BI Developer",
]

class JobType(Enum):
    PROJECT = "project-based"
    PERMANENT = "permanent"

class SearchType(Enum):
    LOOKING_FOR_WORK = "looking for work"
    LOOKING_FOR_EMPLOYEE = "looking for employee"

class State(TypedDict):
    """Agent state for saving info about whole process of classification"""
    description: str
    job_type: str
    category: str
    search_type: str
    processed: bool


In [5]:
#add some instruments

#local read and write
@tool
def load_jobs_data(path: str = "test.json"):
    """load job descriptions from a JSON file"""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"error": f"File '{path}' not found."}

@tool
def append_file(path: str, content: str):
    """Append text to a file."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(content)

# and mcp
mcp_client = MultiServerMCPClient(
        {
            "filesystem": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", "."],
                "transport": "stdio",
            }
        }
    )
mcp_tools = await mcp_client.get_tools()

tools = [load_jobs_data, append_file] + mcp_tools

In [6]:
class VacancyClassificationAgent:
    """main agent method-library"""
    
    def __init__(self, llm, tools):
        self.model_name = (
            getattr(llm, "model", None)
            or getattr(llm, "model_name", None)
            or getattr(llm, "model_id", None)
            or "unknown_model"
        )
        self.tools = {tool.name: tool for tool in tools}
        self.llm = llm.bind_tools(tools)
        self.workflow = self._create_workflow()
    

    def _create_workflow(self) -> StateGraph:
        """building graph structure with LangGraph"""
        workflow = StateGraph(State)

        #add nodes
        workflow.add_node("load_file", self._load_file)
        workflow.add_node("job_type_classification", self._classify_job_type)
        workflow.add_node("category_classification", self._classify_category)
        workflow.add_node("search_type_classification", self._classify_search_type)
        workflow.add_node("save_result", self._save_result)

        #add sequence
        workflow.set_entry_point("load_file")
        workflow.add_edge("load_file", "job_type_classification")
        workflow.add_edge("job_type_classification", "category_classification")
        workflow.add_edge("category_classification", "search_type_classification")
        workflow.add_edge("search_type_classification", "save_result")
        workflow.add_edge("save_result", END)

        return workflow.compile()
    
    def _get_text(self,response):
        if isinstance(response.content, str):
            return response.content

        return "".join(
            block["text"]
            for block in response.content
            if block.get("type") == "text"
        )
    
    async def _classify_job_type(self, state: State) -> dict:
        """node for classification job type"""

        prompt = ChatPromptTemplate.from_messages([
            ("system", """Analyze the following description and determine the job type.
                            Respond with only one of the following two options:
                            - "project job" — if it is a temporary task, project, freelance work, or one-time assignment.
                            - "permanent job" — if it is a full-time position, staff role, or long-term employment.
                            Job type:"""),
            ("human", "{description}")
        ])

        messages = prompt.invoke({"description": state["description"]})
        response = await self.llm.ainvoke(messages)
        job_type = self._get_text(response).strip().lower()
        
        # normalyzing response
        if "project" in job_type or "project-base" in job_type or "freelance" in job_type or "part-time" in job_type:
            job_type = JobType.PROJECT.value
        else:
            job_type = JobType.PERMANENT.value

        return {"job_type": job_type}
    
    async def _classify_category(self, state: State) -> dict:
        """node for classification profession"""
        categories_str = "\n".join([f"- {cat}" for cat in CATEGORIES])

        prompt = ChatPromptTemplate.from_messages([
            ("system", """Analyze the following description and determine the maximum appropriate           
                            profession category. Return EXACTLY one of the categories below, nothing else:
                            {categories_str}"""),
            ("human", "{description}")
        ])
        
        messages = prompt.invoke({"description": state["description"],"categories_str":categories_str})
        response = await self.llm.ainvoke(messages)
        category = self._get_text(response).strip()

        # check category in the list
        if category not in CATEGORIES:
            category = await self._normalize_category(category)

        return {"category": category}
    
    async def _classify_search_type(self, state: State) -> dict:
        """node of classification search type"""

        prompt = ChatPromptTemplate.from_messages([
            ("system", """Analyze the following description and determine who is looking for what.
                            Respond with only one of the following two options:
                            - "looking for work" — if author is looking for work, job, contract, project.
                            - "looking for employee" — if the author is looking for an employee, contractor, or freelancer.
                            Take into account these keywords:
                            - "i am looking for the job", "resume", "want to work" = looking for work
                            - "need", "we are looking for a specialist", "need a specialist", "we're hiring","vacancy", "looking for a developer" = looking for employee
                        Search type:"""),
            ("human", "{description}")
        ])

        messages = prompt.invoke({"description": state["description"]})
        response = await self.llm.ainvoke(messages)
        search_type = self._get_text(response).strip().lower()

        # normalyzing response
        if any(x in search_type for x in [
            "looking for work",
            "job search",
            "job seeker",
            "seeking employment",
            "looking for a position"
        ]):
            search_type = SearchType.LOOKING_FOR_WORK.value
        else:
            search_type = SearchType.LOOKING_FOR_EMPLOYEE.value

        return {"search_type": search_type,
                "processed": True}
    

    async def _normalize_category(self, predicted_category: str) -> str:
        """find the most similar profession category form the list with additional LLM-query"""
        
        categories_str = "\n".join([f"- {cat}" for cat in CATEGORIES])
        prompt = ChatPromptTemplate.from_messages([
            ("system", """Map the predicted profession category to exactly one category from the list.
                        {categories_str}
                        If there is no good semantic match, return EXACTLY:
                        UNKNOWN
                        """),
            ("human", "{predicted_category}")
        ])
        
        messages = prompt.invoke({"predicted_category": predicted_category,"categories_str":categories_str})
        response = await self.llm.ainvoke(messages)
        category = self._get_text(response).strip()
        if category not in CATEGORIES and category != "UNKNOWN":
            category = "UNKNOWN"
        return category
        
    
    async def classify(self, description: str) -> dict:
        """main classification method"""
        initial_state = {
            "description": description,
            "job_type": "",
            "category": "",
            "search_type": "",
            "processed": False
        }

        result = await self.workflow.ainvoke(initial_state)

        # frame the response in JSON
        classification_result = {
            "job_type": result["job_type"],
            "category": result["category"],
            "search_type": result["search_type"],
            "success": result["processed"]
        }

        return classification_result
    
    async def _load_file(self, state: State):
        jobs = await self.tools["load_jobs_data"].ainvoke({"path": "test.json"})
        return {"jobs": jobs}

    async def _save_result(self, state: State):
        result = {
            "model": self.model_name,
            "job_type": state["job_type"],
            "category": state["category"],
            "search_type": state["search_type"],
        }

        await self.tools["append_file"].ainvoke({
            "path": "classification_results.jsonl",
            "content": json.dumps(result, ensure_ascii=False) + "\n"
        })
        return {"processed": True}


In [7]:
# will check agents' graph
with open('test.json', 'r', encoding='utf-8') as file:
    jobs_data = json.load(file)

# check data
for index, job in enumerate(jobs_data):
    print(f"Description {index + 1}: {job['description'][:100]}\n"
          f"Role: {job['role']}\n"
          f"\n")

Description 1: The Data Science Team within Customer Success is highly engaged with our clients making use of their
Role: permanent, Senior Data Scientist, looking for employee


Description 2: Мы ищем сильные руки в нашу команду Octopus, наш элитный R&D-отряд из 4-х человек — лида, двух quant
Role: permanent, Quant(HFT), looking for employee


Description 3: About the job
Descripción específica del perfil
Los chicos diseñan, desarrollan e implementan sistem
Role: permanent, Machine learning expert, looking for employee


Description 4: Descripción del empleo
Estamos buscando chico/a para sumar a nuestro equipo de Operational Fraud Reg
Role: permanent, Operational Fraud Analyst, looking for employee


Description 5: Роль предполагает работу на стороне заказчика в формате outstaff. Конкретные задачи могут зависеть о
Role: permanent, Data Engineer, looking for employee




In [8]:
#build main()
async def main(llm):
    """let's go!"""
    agent = VacancyClassificationAgent(llm, tools)

    print("Let's classify our job data!\n")

    for i, job in enumerate(jobs_data, 1):
        print(f"Test {i}:")
        print(f"Description: {job['description']}")

        try:
            result = await agent.classify(job["description"])
            print(f"Result: {json.dumps(result, ensure_ascii=False, indent=2)}")
            print(f"True_info: {job['role']}")
        except Exception as e:
            print(f"Error: {e}")

        print("-" * 80)


In [9]:
# try with local gemma
await main(ChatOllama(model="gemma4:12b-mlx", temperature=0))

Let's classify our job data!

Test 1:
Description: The Data Science Team within Customer Success is highly engaged with our clients making use of their critical thinking skills with a business-focused mentality and customer-facing attitude. They activate, maintain, and support clients, develop models and rules, and train & enable them. In addition, they work cross-functionally with other departments (e.g., Research, Product, Marketing) in a collaborative team spirit spanning the globe to ensure we deliver best in class risk prevention solutions. Being on the frontline of fighting crime and protecting people from financial harm is incredibly inspiring to each of us. Join Us!
Your Day To Day
Understanding the data which our clients provide to us;
Cleaning that data and validating that it is correct;
Preprocessing the data, usually by using a mixture of shell scripts and a programming language such as Python, Java, Scala, etc
Iteratively computing features and tuning parameters to improve

In [10]:
# try with grok
await main(ChatGroq(model="llama-3.3-70b-versatile",api_key=GROQ_API_KEY,temperature=0,))

Let's classify our job data!

Test 1:
Description: The Data Science Team within Customer Success is highly engaged with our clients making use of their critical thinking skills with a business-focused mentality and customer-facing attitude. They activate, maintain, and support clients, develop models and rules, and train & enable them. In addition, they work cross-functionally with other departments (e.g., Research, Product, Marketing) in a collaborative team spirit spanning the globe to ensure we deliver best in class risk prevention solutions. Being on the frontline of fighting crime and protecting people from financial harm is incredibly inspiring to each of us. Join Us!
Your Day To Day
Understanding the data which our clients provide to us;
Cleaning that data and validating that it is correct;
Preprocessing the data, usually by using a mixture of shell scripts and a programming language such as Python, Java, Scala, etc
Iteratively computing features and tuning parameters to improve

In [11]:
# try with gemini
await main(ChatGoogleGenerativeAI(model="gemini-flash-lite-latest",api_key=GEMINI_API_KEY,temperature=0,))

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


Let's classify our job data!

Test 1:
Description: The Data Science Team within Customer Success is highly engaged with our clients making use of their critical thinking skills with a business-focused mentality and customer-facing attitude. They activate, maintain, and support clients, develop models and rules, and train & enable them. In addition, they work cross-functionally with other departments (e.g., Research, Product, Marketing) in a collaborative team spirit spanning the globe to ensure we deliver best in class risk prevention solutions. Being on the frontline of fighting crime and protecting people from financial harm is incredibly inspiring to each of us. Join Us!
Your Day To Day
Understanding the data which our clients provide to us;
Cleaning that data and validating that it is correct;
Preprocessing the data, usually by using a mixture of shell scripts and a programming language such as Python, Java, Scala, etc
Iteratively computing features and tuning parameters to improve

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported 

Result: {
  "job_type": "permanent",
  "category": "Data Scientist",
  "search_type": "looking for employee",
  "success": true
}
True_info: permanent, Senior Data Scientist, looking for employee
--------------------------------------------------------------------------------
Test 2:
Description: Мы ищем сильные руки в нашу команду Octopus, наш элитный R&D-отряд из 4-х человек — лида, двух quant-researcher'ов и инженера. Ресерчеры — медалисты IPhO с сильным бэкграундом в ML, а инженер — крутой технарь с большой экспертизой в оптимизации и ускорениях.
Меньше чем за год ребята с нуля разработали несколько стратегий, которые торгуют неочевидными сигналами на бирже. Секрет успеха — в комбинации математики, логики, предсказательной силы ML, а также в умении быстро итерироваться и находить нетривиальные закономерности там, где другие их не видят.
Что предстоит делать
Внимательно смотреть в данные и находить аномалии, которые не видны обычному взору (задачка со звездочкой, не так ли?).
Разраб

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported 

...

Result: {
  "job_type": "permanent",
  "category": "Data Scientist",
  "search_type": "looking for employee",
  "success": true
}
True_info: permanent, Machine learning expert, looking for employee
--------------------------------------------------------------------------------
Test 4:
Description: Descripción del empleo
Estamos buscando chico/a para sumar a nuestro equipo de Operational Fraud Regional. Como Operational Fraud Analyst tu rol consistirá en la identificación de oportunidades en la operación detectando comportamiento fraudulento en nuestro ecosistema y desarrollando planes de mitigación.
Identificar nuevos casos de fraude, a partir de la articulación con los equipos de operaciones locales y regionales. 
Analizar y cuantificar patrones de comportamiento para el desarrollo de modelos de prevención de fraude. 
Definir, junto con los equipos de Data y Tecnología los KPIs clave para un correcto monitoreo de la prevención de fraude. 
Monitorear la evolution de las principales mé

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported 

Result: {
  "job_type": "permanent",
  "category": "Data Analyst",
  "search_type": "looking for employee",
  "success": true
}
True_info: permanent, Operational Fraud Analyst, looking for employee
--------------------------------------------------------------------------------
Test 5:
Description: Роль предполагает работу на стороне заказчика в формате outstaff. Конкретные задачи могут зависеть от проекта: от подготовки и обработки данных до разработки ML-моделей, пайплайнов, интеграций и решений для аналитики или автоматизации бизнес-процессов.
Позиция подойдет специалисту, который уверенно работает с Python, данными, ML-инструментами и понимает, как доводить техническое решение до рабочего состояния в продуктовой или проектной среде.
Задачи
Собирать, обрабатывать и подготавливать данные для аналитических и ML-задач.
Разрабатывать и поддерживать ETL / ELT-пайплайны.
Работать с источниками данных, API, базами данных и хранилищами.
Проводить анализ качества данных, находить и исправлят

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported 

Result: {
  "job_type": "project-based",
  "category": "MLOps Engineer",
  "search_type": "looking for employee",
  "success": true
}
True_info: permanent, Data Engineer, looking for employee
--------------------------------------------------------------------------------
